<img align="left" src="https://www.linea.org.br/brand/linea-logo-color.svg" width=100 style="padding: 40px">  
<img align="left" src="https://cdn2.webdamdb.com/1280_c3PXjCZbPM23.png" width=180>

# Rubin DP1 — PZ Compute QA
<font size=4>Lightweight QA of point estimates and N(z)</font>

Run prepared for: **FzBoost / GAaP 1.0 / matched v4**

This notebook performs the basic **PZ compute QA** requested for DP1:

- reads point estimates from the HATS `pz_summary` catalog with LSDB;
- computes global statistics using distributed Dask reductions;
- builds N(z) from `zmode`, `zmean`, and `zmedian`;
- optionally stacks the PDF Parquet shards and compares their N(z) shape with the point-estimate curves.

Only reduced tables and one-dimensional histograms are brought to the notebook process. The complete catalogs and PDF matrices remain partitioned on the workers. This is a compute QA, not the richer PZ validation QA (which would require truth data and metrics such as bias, scatter, and outlier rate).

## 1. Environment

The kernel needs `lsdb`, `dask`, `distributed`, `pandas`, `pyarrow`, and `matplotlib`. For SLURM also install `dask-jobqueue`. No package is installed by this notebook.

In [ ]:
from pathlib import Path
import os
import warnings

import dask
import dask.dataframe as dd
from dask import delayed
from dask.distributed import Client, LocalCluster
from IPython.display import display
import lsdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.6g}")

## 2. Configuration

On Apollo, either replace `RUN_DIR` below or export `PZ_QA_RUN_DIR`. Set `CLUSTER_TYPE` to `"slurm"` there. The defaults run against the local replica when Jupyter starts in the repository root.

In [ ]:
# ---------- Data paths ----------
RUN_DIR = Path(
    os.environ.get(
        "PZ_QA_RUN_DIR",
        "not-tracked/pz-compute-runs/dp1-stage2-fzboost-gaap1p0-matched-v4",
    )
).expanduser().resolve()
# Apollo example:
# RUN_DIR = Path("/scratch/users/luigi.silva/pz-compute-runs/dp1-stage2-fzboost-gaap1p0-matched-v4")

PRODUCT_NAME = "pz-fzboost-gaap1p0-matched-v4-hats"
HATS_COLLECTION_DIR = RUN_DIR / PRODUCT_NAME
SUMMARY_HATS_DIR = HATS_COLLECTION_DIR / "pz_summary"
PDF_DIR = RUN_DIR / f"{PRODUCT_NAME}.pdf"
XVALS_PATH = PDF_DIR / "xvals.parquet"

# ---------- QA choices ----------
POINT_ESTIMATE_COLUMNS = ["zmode", "zmean", "zmedian"]
SUMMARY_COLUMNS = [
    "coord_ra", "coord_dec", "zmode", "zmean",
    "zmedian", "z_p16", "z_p84",
]
RUN_PDF_STACK = True       # Set False for the fastest point-estimate-only QA.
PDF_BATCH_ROWS = 10_000    # Bounds temporary PDF matrix memory inside each worker.

# ---------- Dask cluster ----------
CLUSTER_TYPE = "local"   # "local", "slurm", or "existing"
DASK_SCHEDULER_ADDRESS = None  # Required only when CLUSTER_TYPE == "existing".

LOCAL_N_WORKERS = 2
LOCAL_THREADS_PER_WORKER = 1
LOCAL_MEMORY_LIMIT = "2GiB"
LOCAL_PROCESSES = True

SLURM_MINIMUM_JOBS = 5
SLURM_MAXIMUM_JOBS = 10
SLURM_CORES = 8
SLURM_MEMORY = "16GB"
SLURM_PROCESSES = 1
SLURM_QUEUE = "cpu"
SLURM_ACCOUNT = "hpc-public"
SLURM_INTERFACE = "ib0"
SLURM_WALLTIME = "01:00:00"
SLURM_JOB_EXTRA_DIRECTIVES = ["--propagate"]
SLURM_WAIT_TIMEOUT = 300

In [ ]:
required_paths = [RUN_DIR, SUMMARY_HATS_DIR]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Required path(s) not found. Update RUN_DIR or PZ_QA_RUN_DIR:\n"
        + "\n".join(str(path) for path in missing_paths)
    )

pdf_files = sorted(PDF_DIR.glob("*-pdf-part*.parquet")) if PDF_DIR.exists() else []
path_report = pd.Series(
    {
        "run directory": str(RUN_DIR),
        "HATS summary": str(SUMMARY_HATS_DIR),
        "PDF directory": str(PDF_DIR),
        "PDF shards found": len(pdf_files),
        "stack PDFs": RUN_PDF_STACK,
    },
    name="value",
)
display(path_report.to_frame())

## 3. Start Dask

The same downstream cells work with a local cluster, a SLURM cluster, or an existing scheduler. The final cell closes the client and cluster so adaptive SLURM jobs are released.

In [ ]:
cluster = None

if CLUSTER_TYPE == "local":
    cluster = LocalCluster(
        n_workers=LOCAL_N_WORKERS,
        threads_per_worker=LOCAL_THREADS_PER_WORKER,
        memory_limit=LOCAL_MEMORY_LIMIT,
        processes=LOCAL_PROCESSES,
        dashboard_address=None,
    )
    client = Client(cluster)
elif CLUSTER_TYPE == "slurm":
    from dask_jobqueue import SLURMCluster

    cluster = SLURMCluster(
        n_workers=SLURM_MINIMUM_JOBS * SLURM_PROCESSES,
        queue=SLURM_QUEUE,
        account=SLURM_ACCOUNT,
        cores=SLURM_CORES,
        memory=SLURM_MEMORY,
        processes=SLURM_PROCESSES,
        interface=SLURM_INTERFACE,
        walltime=SLURM_WALLTIME,
        job_extra_directives=SLURM_JOB_EXTRA_DIRECTIVES,
    )
    cluster.adapt(
        minimum_jobs=SLURM_MINIMUM_JOBS,
        maximum_jobs=SLURM_MAXIMUM_JOBS,
    )
    client = Client(cluster)
    client.wait_for_workers(SLURM_MINIMUM_JOBS, timeout=SLURM_WAIT_TIMEOUT)
elif CLUSTER_TYPE == "existing":
    if not DASK_SCHEDULER_ADDRESS:
        raise ValueError("Set DASK_SCHEDULER_ADDRESS for an existing cluster.")
    client = Client(DASK_SCHEDULER_ADDRESS)
else:
    raise ValueError(f"Unsupported CLUSTER_TYPE: {CLUSTER_TYPE!r}")

client

## 4. Point-estimate catalog

LSDB reads the HATS metadata and exposes its spatial partitions as a lazy Dask dataframe. `head` is the only preview operation; no full catalog is materialized in the notebook.

In [ ]:
summary_catalog = lsdb.read_hats(SUMMARY_HATS_DIR)
summary_ddf = summary_catalog.to_dask_dataframe()

missing_columns = sorted(set(SUMMARY_COLUMNS) - set(summary_ddf.columns))
if missing_columns:
    raise KeyError(f"Missing expected HATS columns: {missing_columns}")

print(f"HATS partitions: {summary_catalog.npartitions:,}")
print(f"Columns: {list(summary_ddf.columns)}")
display(summary_catalog.head(5))

### Global statistics

Dask performs the reductions on the workers. Quantiles are approximate and are included for QA characterization; `count`, mean, standard deviation, minimum, and maximum are ordinary distributed reductions. Missing counts are derived from the total row count and each column's non-null count.

In [ ]:
percentiles = [0.01, 0.16, 0.50, 0.84, 0.99]
stats_task = summary_ddf[SUMMARY_COLUMNS].describe(percentiles=percentiles)
count_task = summary_ddf[SUMMARY_COLUMNS].count(axis=0)
row_count_task = summary_ddf.shape[0]

global_stats, non_null_counts, n_objects = dask.compute(
    stats_task, count_task, row_count_task
)
n_objects = int(n_objects)
missing_counts = (n_objects - non_null_counts).astype("int64")

print(f"Objects in HATS: {n_objects:,}")
display(global_stats)
display(pd.DataFrame({"non_null": non_null_counts, "missing": missing_counts}))

### Basic consistency checks

These are inexpensive compute-level checks, not scientific validation. The HATS metadata row count is compared with the distributed count, and the photo-z interval ordering is checked in parallel.

In [ ]:
def read_properties(path):
    values = {}
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            values[key.strip()] = value.strip()
    return values

hats_properties = read_properties(SUMMARY_HATS_DIR / "hats.properties")
expected_rows = int(hats_properties["hats_nrows"])
ordered_intervals = (
    (summary_ddf["z_p16"] <= summary_ddf["zmedian"])
    & (summary_ddf["zmedian"] <= summary_ddf["z_p84"])
)
invalid_interval_count = int((~ordered_intervals).sum().compute())

checks = pd.DataFrame(
    [
        ("HATS row count matches metadata", n_objects == expected_rows, f"{n_objects:,} / {expected_rows:,}"),
        ("Expected columns are present", not missing_columns, str(missing_columns or "all present")),
        ("No missing summary values", int(missing_counts.sum()) == 0, f"{int(missing_counts.sum()):,} missing"),
        ("z_p16 <= zmedian <= z_p84", invalid_interval_count == 0, f"{invalid_interval_count:,} invalid"),
    ],
    columns=["check", "passed", "detail"],
).set_index("check")
display(checks)

## 5. N(z) from HATS point estimates

The grid stored in `xvals.parquet` defines common redshift-bin centers. Each HATS partition produces only three small histograms; they are summed on the notebook process. Thus, the full point-estimate columns are never collected into local memory.

In [ ]:
if not XVALS_PATH.exists():
    raise FileNotFoundError(f"Redshift grid not found: {XVALS_PATH}")

z_grid = pq.read_table(XVALS_PATH, columns=["z"]).column("z").to_numpy()
if len(z_grid) < 2 or not np.all(np.diff(z_grid) > 0):
    raise ValueError("The redshift grid must be strictly increasing.")
bin_edges = np.concatenate(
    ([z_grid[0] - (z_grid[1] - z_grid[0]) / 2],
     (z_grid[:-1] + z_grid[1:]) / 2,
     [z_grid[-1] + (z_grid[-1] - z_grid[-2]) / 2])
)

def histogram_point_partition(frame, columns, edges):
    counts = np.zeros((len(columns), len(edges) - 1), dtype=np.int64)
    outside = np.zeros(len(columns), dtype=np.int64)
    for index, column in enumerate(columns):
        values = frame[column].to_numpy(dtype=np.float64, copy=False)
        finite = values[np.isfinite(values)]
        counts[index] = np.histogram(finite, bins=edges)[0]
        outside[index] = finite.size - counts[index].sum()
    return counts, outside

point_tasks = [
    delayed(histogram_point_partition)(partition, POINT_ESTIMATE_COLUMNS, bin_edges)
    for partition in summary_ddf[POINT_ESTIMATE_COLUMNS].to_delayed()
]
point_results = dask.compute(*point_tasks)
point_histograms = sum((item[0] for item in point_results), start=np.zeros((len(POINT_ESTIMATE_COLUMNS), len(z_grid)), dtype=np.int64))
point_outside_grid = sum((item[1] for item in point_results), start=np.zeros(len(POINT_ESTIMATE_COLUMNS), dtype=np.int64))

display(pd.DataFrame({"inside_grid": point_histograms.sum(axis=1), "outside_grid": point_outside_grid}, index=POINT_ESTIMATE_COLUMNS))

## 6. Optional stacked-PDF N(z)

Each worker reads only the Parquet `pdf` column and reduces it in bounded batches. Every valid PDF is normalized to unit discrete mass before stacking. If the number of PDF rows differs from the HATS object count, the curve is explicitly labeled **partial** and is used only for a normalized shape comparison.

In [ ]:
def stack_pdf_partition(frame, n_bins, batch_rows):
    stacked = np.zeros(n_bins, dtype=np.float64)
    total_rows = len(frame)
    valid_rows = 0
    invalid_rows = 0

    for start in range(0, total_rows, batch_rows):
        values = frame["pdf"].iloc[start:start + batch_rows].to_numpy()
        try:
            matrix = np.stack(values).astype(np.float64, copy=False)
        except ValueError:
            invalid_rows += len(values)
            continue
        if matrix.ndim != 2 or matrix.shape[1] != n_bins:
            invalid_rows += matrix.shape[0]
            continue
        masses = matrix.sum(axis=1)
        valid = np.isfinite(matrix).all(axis=1) & (matrix >= 0).all(axis=1) & (masses > 0)
        if valid.any():
            stacked += (matrix[valid] / masses[valid, None]).sum(axis=0)
        valid_rows += int(valid.sum())
        invalid_rows += int((~valid).sum())

    return stacked, total_rows, valid_rows, invalid_rows

stacked_pdf = None
pdf_total_rows = 0
pdf_valid_rows = 0
pdf_invalid_rows = 0
pdf_is_complete = False

if RUN_PDF_STACK:
    if not pdf_files:
        warnings.warn(f"No PDF Parquet shards found in {PDF_DIR}; skipping PDF stack.")
    else:
        pdf_ddf = dd.read_parquet(
            [str(path) for path in pdf_files],
            columns=["pdf"],
            split_row_groups=True,
        )
        pdf_tasks = [
            delayed(stack_pdf_partition)(partition, len(z_grid), PDF_BATCH_ROWS)
            for partition in pdf_ddf.to_delayed()
        ]
        pdf_results = dask.compute(*pdf_tasks)
        stacked_pdf = sum((item[0] for item in pdf_results), start=np.zeros(len(z_grid), dtype=np.float64))
        pdf_total_rows = sum(item[1] for item in pdf_results)
        pdf_valid_rows = sum(item[2] for item in pdf_results)
        pdf_invalid_rows = sum(item[3] for item in pdf_results)
        pdf_is_complete = pdf_total_rows == n_objects

        pdf_report = pd.Series(
            {
                "Parquet shards": len(pdf_files),
                "PDF rows": pdf_total_rows,
                "valid PDF rows": pdf_valid_rows,
                "invalid PDF rows": pdf_invalid_rows,
                "HATS rows": n_objects,
                "PDF product complete": pdf_is_complete,
            },
            name="value",
        )
        display(pdf_report.to_frame())
        if not pdf_is_complete:
            warnings.warn(
                "PDF rows do not match the HATS row count. The stacked-PDF curve below "
                "represents only the available shards; use it as a shape diagnostic."
            )
else:
    print("PDF stacking disabled; point-estimate N(z) is still available.")

## 7. N(z) plots

In [ ]:
colors = {"zmode": "#0072B2", "zmean": "#D55E00", "zmedian": "#009E73"}
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

for index, column in enumerate(POINT_ESTIMATE_COLUMNS):
    axes[0].step(z_grid, point_histograms[index], where="mid", label=column, color=colors[column], linewidth=1.8)
    denominator = point_histograms[index].sum()
    if denominator:
        axes[1].step(z_grid, point_histograms[index] / denominator, where="mid", label=column, color=colors[column], linewidth=1.6)

if stacked_pdf is not None and pdf_valid_rows:
    completeness = "complete" if pdf_is_complete else "partial"
    axes[1].plot(
        z_grid, stacked_pdf / stacked_pdf.sum(),
        color="black", linewidth=2.2, alpha=0.8,
        label=f"stacked PDFs ({completeness}; n={pdf_valid_rows:,})",
    )

axes[0].set(title="Point-estimate N(z)", xlabel="Photometric redshift", ylabel="Objects per bin")
axes[1].set(title="Normalized N(z) shape", xlabel="Photometric redshift", ylabel="Fraction per bin")
for axis in axes:
    axis.set_xlim(bin_edges[0], bin_edges[-1])
    axis.legend()
plt.show()

## 8. QA summary and cleanup

In [ ]:
final_summary = pd.Series(
    {
        "HATS objects": n_objects,
        "HATS partitions": summary_catalog.npartitions,
        "basic checks passed": f"{int(checks['passed'].sum())}/{len(checks)}",
        "point estimates outside PDF grid": int(point_outside_grid.sum()),
        "PDF stacking requested": RUN_PDF_STACK,
        "valid PDF rows used": pdf_valid_rows,
        "PDF coverage complete": pdf_is_complete if RUN_PDF_STACK and pdf_files else "not evaluated",
    },
    name="value",
)
display(final_summary.to_frame())

if not checks["passed"].all():
    warnings.warn("One or more basic QA checks failed; inspect the tables above.")

In [ ]:
client.close()
if cluster is not None:
    cluster.close()
print("Dask resources released.")

### Acknowledgments

*This notebook was supported by resources supplied by the Laboratório Interinstitucional de e-Astronomia (LIneA).*